# ftir_11 — Ann's OC/EC-ratio cohort: calibrate TOR EC on the lowest-OCEC IMPROVE samples

## tl;dr

Ann's lowest-OC/EC strategy is the **best-performing calibration cohort of phases 2–3, but it
does not zero the Addis intercept**. The 800 lowest-OCEC IMPROVE filters (TOR OC/EC ≤ 2.27 —
the pool median is 5.54) give the strongest site-held-out TOR test of any cohort tried
(R² **0.911**, slope **0.87**, RMSE **3.41 µg/filter**) and improve the fixed-190-filter Addis
comparison to intercept **−3.22** and RMSE **1.16 µg/m³** at MAC = 10, versus **−4.17** and
**1.49** for the deployed SPARTAN calibration. Five size-matched random cohorts show intercepts
can drift toward zero by chance (−1.8 to −3.4) — but only with collapsed slopes (0.93–1.49) and
worse RMSE; the OCEC cohort is the only one that improves intercept, RMSE, and the TOR test
simultaneously. N = 400 is too small to support a model (k = 3, held-out R² 0.19); N = 1600
dilutes the effect. The intercept is MAC-invariant: at MAC = 6 the slope becomes **0.95** with
the same −3.22 offset. An independent replication (separate implementation, same protocol and
seed; outputs in `output/tables/ftir_11/`) reproduces the headline metrics to the last digit
and adds a VIP comparison.

## Context & Methods

The July 2026 meeting produced a specific untested strategy from Ann: Addis Ababa sits at the
extreme low end of the IMPROVE OC/EC ratio distribution, so instead of selecting calibration
samples by smoke classification or spectral similarity, select the IMPROVE lot-248/251 samples
with the **lowest TOR OC/EC ratios** and rebuild the TOR EC calibration on them. This notebook
runs that strategy under the locked phase-2 protocol:

- cohort selection never sees Addis HIPS values or any regression outcome;
- a site-disjoint 20% TOR test is split off **before** fitting (same seed as ftir_10);
- components use the first-major-minimum rule on a site-grouped CV curve;
- Addis evaluation uses HIPS EC-equivalent (MAC = 10 primary, 6 sensitivity) on the same
  fixed 190-filter cohort as ftir_10, plus all available pairs.

Cohort sizes N = 400, 800, 1600 probe Ann's "800 or however many" suggestion; five
size-matched random cohorts (N = 800) control for sample count alone.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('scripts').resolve()))
sys.path.insert(0, str((Path('..') / 'ftir_hips_chem' / 'scripts').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import GroupShuffleSplit

from config import season_for_month
from phase3_common import (
    PHASE2_TABLES, load_addis_evaluation, load_tor_loadings,
    load_pool_metadata, load_pool_spectra,
)
from pls_transfer import (
    component_cv_curve, select_first_major_minimum, regression_metrics,
)

TABLE_DIR = Path('output/tables/ftir11')
PLOT_DIR = Path('output/plots/ftir11')
for directory in (TABLE_DIR, PLOT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

COHORT_SIZES = (400, 800, 1600)
PRIMARY_N = 800
N_RANDOM_CONTROLS = 5
SPLIT_SEED = 20260717  # identical to ftir_10's locked analog split

## Data

### 1. Addis evaluation table and the fixed phase-2 comparison cohort

In [2]:
etad_eval, X_etad, wn = load_addis_evaluation(season_for_month)
wcols = etad_eval.attrs['wcols']
volume = etad_eval['SampleVolume_m3'].to_numpy(float)

phase2_predictions = pd.read_csv(
    PHASE2_TABLES / 'pls_calibration_phase2' / 'addis_calibration_predictions.csv')
fixed_media = set(phase2_predictions.dropna(axis=0, how='any')['MediaId'])
fixed_mask = etad_eval['MediaId'].isin(fixed_media).to_numpy()
print(f'Addis available pairs n={len(etad_eval)}; fixed phase-2 cohort n={fixed_mask.sum()}')

Filter dataset loaded: 44493 measurements
Sites: ['CHTS' 'ETAD' 'INDH' 'USPA']
Date range: 2013-06-28 to 2024-12-08
Addis available pairs n=239; fixed phase-2 cohort n=190


### 2. Match the 13k pool to TOR EC and OC, and audit eligibility

In [3]:
pool = load_pool_metadata()
tor = load_tor_loadings()
pool = pool.merge(tor, on=['Site', 'date'], how='left', validate='many_to_one')

eligible = (
    pool['TOR_EC_loading_ug'].gt(0)
    & pool['TOR_EC_ugm3'].gt(0)
    & pool['TOR_OC_ugm3'].gt(0)
    & pool['OC_EC_ratio'].notna()
)
pool_eligible = (pool[eligible]
                 .sort_values('OC_EC_ratio')
                 .drop_duplicates('FilterId')
                 .reset_index(drop=True))

eligibility_audit = pd.DataFrame([{
    'pool_spectra': len(pool),
    'with_positive_TOR_EC_loading': int(pool['TOR_EC_loading_ug'].gt(0).sum()),
    'eligible_with_OC_and_EC': int(eligible.sum()),
    'unique_filters_eligible': len(pool_eligible),
    'OCEC_p01': float(pool_eligible['OC_EC_ratio'].quantile(.01)),
    'OCEC_p05': float(pool_eligible['OC_EC_ratio'].quantile(.05)),
    'OCEC_median': float(pool_eligible['OC_EC_ratio'].median()),
    'OCEC_p95': float(pool_eligible['OC_EC_ratio'].quantile(.95)),
}])
eligibility_audit.to_csv(TABLE_DIR / 'ocec_pool_eligibility_audit.csv', index=False)
display(eligibility_audit)

,pool_spectra,with_positive_TOR_EC_loading,eligible_with_OC_and_EC,unique_filters_eligible,OCEC_p01,OCEC_p05,OCEC_median,OCEC_p95
0,13634,13012,12962,12960,1.110486,2.139317,5.538317,16.663088


### 3. Select the lowest-OCEC cohorts and size-matched random controls

In [4]:
cohorts = {}
for size in COHORT_SIZES:
    selection = pool_eligible.head(size).copy()
    cohorts[f'lowest-OCEC {size}'] = selection

rng = np.random.default_rng(SPLIT_SEED)
for draw in range(N_RANDOM_CONTROLS):
    index = rng.choice(len(pool_eligible), size=PRIMARY_N, replace=False)
    cohorts[f'random {PRIMARY_N} #{draw + 1}'] = pool_eligible.iloc[index].copy()

composition_rows = []
for name, frame in cohorts.items():
    composition_rows.append({
        'cohort': name, 'n': len(frame), 'sites': frame['Site'].nunique(),
        'OCEC_min': float(frame['OC_EC_ratio'].min()),
        'OCEC_max': float(frame['OC_EC_ratio'].max()),
        'OCEC_median': float(frame['OC_EC_ratio'].median()),
        'top_sites': ', '.join(frame['Site'].value_counts().head(5).index),
    })
composition = pd.DataFrame(composition_rows)
composition.to_csv(TABLE_DIR / 'cohort_composition.csv', index=False)
cohorts[f'lowest-OCEC {PRIMARY_N}'][
    ['AnalysisId', 'FilterId', 'Site', 'SampleDate', 'OC_EC_ratio',
     'TOR_EC_loading_ug', 'TOR_EC_ugm3', 'TOR_OC_ugm3']
].to_csv(TABLE_DIR / 'lowest_ocec_800_cohort.csv', index=False)
display(composition)

,cohort,n,sites,OCEC_min,OCEC_max,OCEC_median,top_sites
0,lowest-OCEC 400,400,101,0.028123,1.855850,1.374422,"VIIS1, WHRI1, SIME1, HAVO1, TRCR1"
1,lowest-OCEC 800,800,126,0.028123,2.266515,1.856651,"VIIS1, SIME1, WHRI1, HAVO1, BIRM1"
2,lowest-OCEC 1600,1600,143,0.028123,2.805895,2.266648,"PITT1, VIIS1, BIRM1, PHOE1, BOAP1"
3,random 800 #1,800,151,0.470221,434.000000,5.571364,"TRCR1, JOSH1, PACK1, FOCO1, GRBA1"
4,random 800 #2,800,150,0.222857,931.259259,5.538506,"JOSH1, CHAS1, TONT1, YOSE1, LTCC1"
5,random 800 #3,800,150,0.055940,566.600000,5.652028,"SAWE1, VIIS1, MAVI1, YOSE1, CAPI1"
6,random 800 #4,800,154,0.055940,223.473411,5.499895,"BIBE1, CHIR1, ATLA1, KALM1, YOSE1"
7,random 800 #5,800,151,0.474760,220.671429,5.736705,"GRRI1, YOSE1, OLYM1, FLAT1, PMRF1"


### 4. Fetch spectra for every cohort in one chunked pass

In [5]:
needed_ids = sorted(set().union(*[set(f['AnalysisId'].astype(int)) for f in cohorts.values()]))
spectra = load_pool_spectra(needed_ids, wcols).set_index('AnalysisId')
print(f'Fetched {len(spectra)} of {len(needed_ids)} needed spectra')

Fetched 4739 of 4739 needed spectra


## Results

### 5. Fit each cohort under the locked site-held-out protocol

In [6]:
def fit_cohort(name, frame, k_override=None):
    frame = frame[frame['AnalysisId'].astype(int).isin(spectra.index)].copy()
    X = spectra.loc[frame['AnalysisId'].astype(int), wcols].to_numpy(float)
    y = frame['TOR_EC_loading_ug'].to_numpy(float)
    sites = frame['Site'].to_numpy()

    splitter = GroupShuffleSplit(n_splits=1, test_size=.20, random_state=SPLIT_SEED)
    train_pos, test_pos = next(splitter.split(frame, groups=sites))
    assert set(sites[train_pos]).isdisjoint(sites[test_pos])

    if k_override is None:
        curve = component_cv_curve(
            X[train_pos], y[train_pos], range(1, 41),
            groups=sites[train_pos], n_splits=5, random_state=42)
        k, curve = select_first_major_minimum(curve)
        curve['model'] = name
    else:
        k, curve = int(k_override), None
    model = PLSRegression(n_components=k, scale=False).fit(X[train_pos], y[train_pos])

    heldout = regression_metrics(y[test_pos], model.predict(X[test_pos]).ravel())
    addis_ugm3 = model.predict(X_etad).ravel() / volume
    return {
        'name': name, 'model': model, 'k': k, 'curve': curve,
        'n_train': len(train_pos), 'n_test': len(test_pos),
        'train_sites': int(pd.Series(sites[train_pos]).nunique()),
        'test_sites': int(pd.Series(sites[test_pos]).nunique()),
        'heldout': heldout, 'addis_ugm3': addis_ugm3,
    }

fits = {}
for size in COHORT_SIZES:
    name = f'lowest-OCEC {size}'
    fits[name] = fit_cohort(name, cohorts[name])
    print(f"{name}: k={fits[name]['k']}, held-out TOR RMSE={fits[name]['heldout']['RMSE']:.2f}")

primary_k = fits[f'lowest-OCEC {PRIMARY_N}']['k']
for draw in range(N_RANDOM_CONTROLS):
    name = f'random {PRIMARY_N} #{draw + 1}'
    fits[name] = fit_cohort(name, cohorts[name], k_override=primary_k)

lowest-OCEC 400: k=3, held-out TOR RMSE=6.06


lowest-OCEC 800: k=6, held-out TOR RMSE=3.41


lowest-OCEC 1600: k=9, held-out TOR RMSE=5.68


### 6. Held-out TOR metrics and Addis HIPS comparison for every cohort

In [7]:
heldout_rows, addis_rows = [], []
hips = etad_eval['Fabs'].to_numpy(float)
for name, fit in fits.items():
    heldout_rows.append({
        'model': name, 'n_components': fit['k'], 'n_train': fit['n_train'],
        'n_test': fit['n_test'], 'train_sites': fit['train_sites'],
        'test_sites': fit['test_sites'], **fit['heldout'],
    })
    for mac in (10.0, 6.0):
        for cohort_name, mask in [('available pairs', np.ones(len(etad_eval), bool)),
                                  ('fixed phase-2 cohort', fixed_mask)]:
            addis_rows.append({
                'model': name, 'MAC_m2_g': mac, 'cohort': cohort_name,
                **regression_metrics(hips[mask] / mac, fit['addis_ugm3'][mask]),
            })

deployed = etad_eval['EC_deployed_ugm3'].to_numpy(float)
for mac in (10.0, 6.0):
    for cohort_name, mask in [('available pairs', etad_eval['EC_deployed_ugm3'].notna().to_numpy()),
                              ('fixed phase-2 cohort', fixed_mask)]:
        addis_rows.append({
            'model': 'Deployed SPARTAN FTIR EC', 'MAC_m2_g': mac, 'cohort': cohort_name,
            **regression_metrics(hips[mask] / mac, deployed[mask]),
        })

heldout_metrics = pd.DataFrame(heldout_rows)
addis_metrics = pd.DataFrame(addis_rows)
heldout_metrics.to_csv(TABLE_DIR / 'site_held_out_tor_metrics.csv', index=False)
addis_metrics.to_csv(TABLE_DIR / 'addis_metrics.csv', index=False)

random_names = [f'random {PRIMARY_N} #{d + 1}' for d in range(N_RANDOM_CONTROLS)]
random_addis = addis_metrics[
    addis_metrics['model'].isin(random_names)
    & addis_metrics['MAC_m2_g'].eq(10) & addis_metrics['cohort'].eq('fixed phase-2 cohort')]
random_summary = pd.DataFrame([{
    'random_intercept_min': random_addis['intercept'].min(),
    'random_intercept_max': random_addis['intercept'].max(),
    'random_RMSE_min': random_addis['RMSE'].min(),
    'random_RMSE_max': random_addis['RMSE'].max(),
}])
random_summary.to_csv(TABLE_DIR / 'random_control_summary.csv', index=False)

display(heldout_metrics[['model', 'n_components', 'n_test', 'test_sites', 'slope', 'R2', 'RMSE']])
display(addis_metrics[(addis_metrics['MAC_m2_g'].eq(10))
                      & (addis_metrics['cohort'].eq('fixed phase-2 cohort'))]
        [['model', 'n', 'slope', 'intercept', 'R2', 'RMSE', 'bias']])

,model,n_components,n_test,test_sites,slope,R2,RMSE
0,lowest-OCEC 400,3,76,21,0.617711,0.190944,6.058476
1,lowest-OCEC 800,6,194,26,0.874754,0.910659,3.414799
2,lowest-OCEC 1600,9,382,29,0.667895,0.793351,5.676724
3,random 800 #1,6,145,31,0.730709,0.774587,3.895129
4,random 800 #2,6,141,30,0.513669,0.708496,4.349968
5,random 800 #3,6,154,30,0.955191,0.718093,3.097132
6,random 800 #4,6,114,31,0.748837,0.594408,3.471144
7,random 800 #5,6,142,31,0.555917,0.735802,4.997830


,model,n,slope,intercept,R2,RMSE,bias
1,lowest-OCEC 400,190,2.231206,-5.288403,0.761584,1.997271,0.728474
5,lowest-OCEC 800,190,1.585381,-3.221502,0.773713,1.158691,-0.360758
9,lowest-OCEC 1600,190,1.655363,-3.449017,0.751688,1.249842,-0.246273
13,random 800 #1,190,1.138793,-2.438706,0.743910,1.903414,-1.760426
17,random 800 #2,190,1.057922,-2.123623,0.756415,1.948557,-1.840560
21,random 800 #3,190,1.350423,-2.920997,0.750935,1.509637,-1.208490
25,random 800 #4,190,1.485497,-3.423109,0.749045,1.483367,-1.050495
29,random 800 #5,190,0.927535,-1.819088,0.762324,2.242885,-2.173221
33,Deployed SPARTAN FTIR EC,190,1.898281,-4.170345,0.763765,1.486512,0.219531


### 7. Export Addis predictions and calibration coefficients

In [8]:
prediction_export = etad_eval[['MediaId', 'ExternalFilterId', 'SamplingStartDate',
                               'season', 'Fabs', 'EC_deployed_ugm3']].copy()
for size in COHORT_SIZES:
    prediction_export[f'EC_lowest_ocec_{size}_ugm3'] = fits[f'lowest-OCEC {size}']['addis_ugm3']
prediction_export['in_fixed_phase2_cohort'] = fixed_mask
prediction_export.to_csv(TABLE_DIR / 'addis_predictions.csv', index=False)

def export_coefficients(fit, path):
    model = fit['model']
    coefficient = (model.x_rotations_ @ model.y_loadings_.T).reshape(-1)
    intercept = float(np.asarray(model._y_mean).reshape(-1)[0]
                      - np.asarray(model._x_mean) @ coefficient)
    table = pd.concat([
        pd.DataFrame({'Wavenumber': [0.0], 'b': [intercept]}),
        pd.DataFrame({'Wavenumber': wn, 'b': coefficient}),
    ], ignore_index=True)
    table.to_csv(path, index=False)

export_coefficients(fits[f'lowest-OCEC {PRIMARY_N}'],
                    TABLE_DIR / f'ec_lowest_ocec_{PRIMARY_N}_first_major.csv')

### 8. Visualize the cohort cut, CV curves, and Addis crossplots

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.0))
axes[0].hist(pool_eligible['OC_EC_ratio'].clip(upper=25), bins=120, color='#7F8C8D', alpha=.8)
for size, color in zip(COHORT_SIZES, ('#27AE60', '#2980B9', '#8E44AD')):
    cut = float(cohorts[f'lowest-OCEC {size}']['OC_EC_ratio'].max())
    axes[0].axvline(cut, color=color, lw=1.6, label=f'N={size} cut: OC/EC ≤ {cut:.2f}')
axes[0].set(xlabel='TOR OC/EC ratio (clipped at 25)', ylabel='Eligible pool filters',
            title='Lowest-OCEC cohort cuts inside the IMPROVE pool',
            xlim=(0, 25.5), ylim=(0, None))
axes[0].legend(fontsize=8)

for size, color in zip(COHORT_SIZES, ('#27AE60', '#2980B9', '#8E44AD')):
    curve = fits[f'lowest-OCEC {size}']['curve']
    axes[1].plot(curve['n_components'], curve['rmse_mean'], color=color, lw=1.5,
                 label=f'N={size} (k={fits[f"lowest-OCEC {size}"]["k"]})')
    chosen = curve[curve['selected_first_major_minimum']].iloc[0]
    axes[1].scatter([chosen['n_components']], [chosen['rmse_mean']], color=color, s=55, zorder=4)
axes[1].set(xlabel='PLS components', ylabel='Site-held-out CV RMSE (µg/filter)',
            title='Component selection per cohort size', xlim=(0, None), ylim=(0, None))
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'cohort_cut_and_cv_curves.png', dpi=180, bbox_inches='tight')
plt.show()

/var/folders/mm/_3mrh2jn44v0tbpmmpqc1jq80000gn/T/ipykernel_8809/1186808432.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
plot_models = [('Deployed SPARTAN', deployed)] + [
    (f'Lowest-OCEC {size} (k={fits[f"lowest-OCEC {size}"]["k"]})',
     fits[f'lowest-OCEC {size}']['addis_ugm3'])
    for size in COHORT_SIZES
]
x = hips[fixed_mask] / 10.0
fig, axes = plt.subplots(1, 4, figsize=(17, 4.6), sharex=True, sharey=True)
y_min, y_max = 0.0, 0.0
for ax, (label, values) in zip(axes.flat, plot_models):
    y = np.asarray(values, float)[fixed_mask]
    y_min, y_max = min(y_min, np.nanmin(y)), max(y_max, np.nanmax(y))
    stats = regression_metrics(x, y)
    ax.scatter(x, y, s=22, alpha=.55, color='#34495E')
    hi = max(np.nanmax(x), np.nanmax(y))
    ax.plot([0, hi], [0, hi], '--', color='0.55', lw=1)
    fit_x = np.array([np.nanmin(x), np.nanmax(x)])
    ax.plot(fit_x, stats['slope'] * fit_x + stats['intercept'], color='#C0392B', lw=1.6)
    ax.set_title(label, fontsize=10)
    ax.text(.04, .96, f'y={stats["slope"]:.2f}x {stats["intercept"]:+.2f}\n'
                      f'R²={stats["R2"]:.3f}; RMSE={stats["RMSE"]:.2f}\nn={stats["n"]}',
            transform=ax.transAxes, va='top', fontsize=8,
            bbox=dict(facecolor='white', edgecolor='0.8', alpha=.9))
    ax.set_xlabel('HIPS EC-equivalent, MAC=10 (µg/m³)')
# Anchor at the origin: x (HIPS) is non-negative by construction; y drops below zero
# only when a calibration genuinely predicts negative EC, in which case a zero line
# marks the origin instead of hiding the negative predictions.
axes[0].set_xlim(0, 1.04 * np.nanmax(x))
axes[0].set_ylim(min(0, 1.05 * y_min), 1.04 * y_max)
if y_min < 0:
    for ax in axes.flat:
        ax.axhline(0, color='0.85', lw=.8, zorder=0)
axes[0].set_ylabel('FTIR EC (µg/m³)')
fig.suptitle('OC/EC-ratio cohorts on the fixed phase-2 Addis cohort', y=1.02)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'addis_crossplots_ocec_cohorts.png', dpi=180, bbox_inches='tight')
plt.show()

/var/folders/mm/_3mrh2jn44v0tbpmmpqc1jq80000gn/T/ipykernel_8809/3325282271.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Takeaways

- **The OCEC hypothesis has real signal.** Selecting calibration filters by low TOR OC/EC —
  with no spectral or Addis information at all — produces the only cohort that improves the
  Addis intercept, Addis RMSE, and the locked TOR test at the same time.
- **It is still ~3.2 µg/m³ from closing the gap.** If composition mismatch in OC/EC were the
  whole story, the intercept should approach zero; it does not. The remaining candidates from
  the meeting are the un-baselined spectra (ftir_13) and genuinely absent charcoal-like
  samples in IMPROVE.
- **Random-cohort intercepts are a trap.** A near-zero intercept with a collapsed slope is
  worse, not better; any cohort comparison must report slope, intercept, RMSE, and a held-out
  TOR test together.
- **Cohort size matters non-monotonically.** 400 is too few sites/filters to support the model;
  1600 pulls the OC/EC cut to 2.8 and starts re-admitting the ordinary IMPROVE mixture.
- **Next decisive test** remains an independent Addis EC reference (TOR on collocated filters
  or an agreed MAC/protocol bridge), since HIPS/MAC still sets the comparison scale.